# 01 — Data audit and cleaning

## Question

Is the Caltech ACN-Data public dataset a legitimate source of empirical behavioral uncertainty for an EV-charging scheduling experiment?

## Why this test exists

If the data is the wrong shape, the wrong fields, the wrong time window, or the wrong scale, every downstream conclusion is suspect. We need to establish, before any QUBO is written, that the data can in principle answer our research question.

## Method

Two-pronged audit:

1. **Static-snapshot audit** (token-independent). The Caltech public static snapshot at `tongxin-li/ACN-Data-Static` contains time-series CSV files but no `userInputs[*]` block. The token-gated live API contains the `userInputs[*].kWhRequested` and `userInputs[*].requestedDeparture` fields that the methodology requires.
2. **Cleaning rules.** The cleaning spec `docs/cleaning_spec.md` defines 11 deterministic rules R1-R11. R1-R8 operate on the static time-series files (filename parse, non-empty body, timestamp parse, energy column present, filename/file timestamp agreement, duration sanity, monotonic timestamps, valid EVSE). R9-R11 operate per-uncertainty-variable on the live API session-level data.

The cleaning rules are pure functions of session-level fields. They do not depend on summary statistics, distributions, or held-out data. This is the leakage-safe-by-construction property that the Stage 2 audit verified.

## Implementation

The cleaning rules are encoded in `artifacts/cleaning_rules.json` and applied by `stage5/uncertainty.py::apply_cleaning_to_live` (R9-R11, per-uncertainty-variable) and the upstream time-series pipeline (R1-R8, performed by the data-acquisition stage when the snapshot is processed).

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


In [ ]:
import json, sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage5.uncertainty import apply_cleaning_to_live, UncertaintySample

rules = json.loads(pathlib.Path('../artifacts/cleaning_rules.json').read_text())
print('R1-R11 cleaning rules:')
for r in rules['rules']:
    print(f"  {r['id']:25s}  on_fail={r['on_fail']:8s}  det={r['deterministic']}")
print()
print('Explicitly NOT excluded (these are real behavior, not data errors):')
for x in rules['explicitly_not_excluded']:
    print(f'  - {x}')


## Result

On a 200-file audit sample of the static snapshot, 8 files were excluded by R1-R8 (4.0% exclusion rate). Projected to the full inventory, this gives approximately 26,690 calibration and 10,525 held-out sessions. **R9-R11 cannot be audited on the static snapshot** because they require the token-gated `userInputs[*]` fields; they are applied at live-acquisition time.

## Interpretation

The cleaning rules are deterministic, leakage-safe, and behave as specified. R1-R8 are upstream of the live API and have been audited on the static snapshot. R9-R11 will be applied by `stage5/uncertainty.py::apply_cleaning_to_live` after the live API is acquired (see notebook 10).

## Limitations

- The static snapshot's `caltech_sessions.json` is **2 bytes (empty)**.   The token-gated live API is the only source of `userInputs[*]`.   Until the token arrives, R9-R11 cannot be exercised on real data.
- The exclusion rate (4.0%) is from a 200-file audit sample, not the   full inventory. A wider audit could reveal edge cases not seen in   the sample.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
